In [290]:
import numpy as np
import astropy.io.ascii
import pandas as pd
import matplotlib.pyplot as plt
from astropy.table import Table
from io import StringIO
from astropy import units as u

In [291]:
df = pd.read_csv("../data/output/outflow_data.csv")
sep = pd.read_csv('../data/output/separations.csv')
notes = pd.read_csv('../data/input/notes.csv')

In [292]:
def getIdx(listOf):
    ranges = []
    
    for string in listOf:
        if pd.isna(string):  # Skip NaN values
            continue
        
        for part in string.split(', '):
            if '-' in part:  # Handle ranges
                start, end = map(int, part.split('-'))
                ranges.append(np.r_[start:end + 1])  # Use np.r_
            else:  # Handle single values
                ranges.append(int(part))  # Append as integer
    
    # Ensure `ranges` is not empty before passing to np.r_
    if not ranges:
        return np.r_[:]  # Returns an empty np.r_

    return np.r_[tuple(ranges)]  # Use `tuple(ranges)` to avoid errors

df.loc[df['outflow_source'] == 'both', 'outflow_source'] = df.loc[df['outflow_source'] == 'both']['source_a'] + '+' + df.loc[df['outflow_source'] == 'both']['source_b']
df['integrated_channels'] = (df['red_channels'].fillna('') + ', ' + df['blue_channels'].fillna('')).str.lstrip(', ').str.rstrip(', ')
df['num_integrated_channels'] = df.apply(
    lambda row: len(getIdx([row['red_channels'], row['blue_channels']])), axis=1
)

In [293]:
by_outflow = df[['field', 'outflow_source', 'outflow_PA', 'binary_PA', 'delta_PA']]
by_outflow['tertiary'] = by_outflow['binary_PA'].isna()

by_outflow = pd.merge(by_outflow, sep, left_on=['field', 'tertiary'], right_on=['pair', 'tertiary'], how='left')
by_outflow['type'] = by_outflow['tertiary'].map({True: 'Tertiary', False: 'Binary'})
by_outflow = by_outflow.drop(['tertiary', 'pair'],axis=1).reset_index(drop=True)
by_outflow = by_outflow[['field', 'type', 'separation', 'class', 'outflow_source', 'outflow_PA', 'binary_PA', 'delta_PA']]
by_outflow = by_outflow.fillna('-')
by_outflow[['outflow_PA', 'binary_PA', 'delta_PA']] = by_outflow[['outflow_PA', 'binary_PA', 'delta_PA']].replace('-', np.nan).astype(float).round(0).astype('Int64')
by_outflow

/var/folders/41/_gkgvhb94wd4156zplzr4cg00000gn/T/ipykernel_30009/769593483.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  by_outflow['tertiary'] = by_outflow['binary_PA'].isna()
/var/folders/41/_gkgvhb94wd4156zplzr4cg00000gn/T/ipykernel_30009/769593483.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  by_outflow[['outflow_PA', 'binary_PA', 'delta_PA']] = by_outflow[['outflow_PA', 'binary_PA', 'delta_PA']].replace('-', np.nan).astype(float).round(0).astype('Int64')


,field,type,separation,class,outflow_source,outflow_PA,binary_PA,delta_PA
0,HOPS-12,Binary,80.6,C0,B-A,337,45,68
1,HOPS-12,Tertiary,1853.5,C0,A,337,<NA>,<NA>
2,HOPS-290,Binary,332.7,C0,A,114,298,5
3,HOPS-290,Binary,332.7,C0,B,49,298,70
4,HOPS-92,Binary,108.4,FS,A-A+A-B,256,209,47
5,HOPS-92,Tertiary,498.8,FS,B,89,<NA>,<NA>
6,HOPS-288,Binary,61.3,C0,A-A+A-B,236,150,86
7,HOPS-288,Tertiary,221.9,C0,B,32,<NA>,<NA>
8,HOPS-203,Binary,49.7,C0,A+B,324,201,57
9,HOPS-203,Tertiary,1085.5,C0,C,281,<NA>,<NA>


In [294]:
t = Table.from_pandas(by_outflow)
buf = StringIO()
t.write(buf, format='ascii.latex', latexdict={'tabletype': 'deluxetable'})
print(buf.getvalue())

\begin{deluxetable}
\begin{tabular}{cccccccc}
field & type & separation & class & outflow_source & outflow_PA & binary_PA & delta_PA \\
HOPS-12 & Binary & 80.6 & C0 & B-A & 337 & 45 & 68 \\
HOPS-12 & Tertiary & 1853.5 & C0 & A & 337 &  &  \\
HOPS-290 & Binary & 332.7 & C0 & A & 114 & 298 & 5 \\
HOPS-290 & Binary & 332.7 & C0 & B & 49 & 298 & 70 \\
HOPS-92 & Binary & 108.4 & FS & A-A+A-B & 256 & 209 & 47 \\
HOPS-92 & Tertiary & 498.8 & FS & B & 89 &  &  \\
HOPS-288 & Binary & 61.3 & C0 & A-A+A-B & 236 & 150 & 86 \\
HOPS-288 & Tertiary & 221.9 & C0 & B & 32 &  &  \\
HOPS-203 & Binary & 49.7 & C0 & A+B & 324 & 201 & 57 \\
HOPS-203 & Tertiary & 1085.5 & C0 & C & 281 &  &  \\
HH270VLA1 & Binary & 99.4 & C0 & A+B & 230 & 124 & 74 \\
HOPS-32 & Binary & 162.2 & C0 & B & 339 & 233 & 74 \\
HOPS-84 & Binary & 275.5 & C1 & A+B & 263 & 209 & 53 \\
HOPS-168 & Binary & 73.7 & C0 & A+B & 339 & 59 & 79 \\
HOPS-281 & Binary & 311.6 & FS & A & 206 & 253 & 47 \\
HOPS-312 & Binary & 134.7 & C0 & A+B & 36 &

In [295]:
by_field = df.groupby(['field']).agg('first').reset_index()[['field', 'source_a_ra', 'source_a_dec', 'integrated_channels', 'num_integrated_channels']]
notes = notes.groupby('Field').agg('first').reset_index()[['Field', 'N', 'V_sys']].dropna()
by_field = pd.merge(by_field, notes, left_on='field', right_on='Field', how='left')
by_field = by_field.rename(columns={'source_a_ra': 'ra', 'source_a_dec': 'dec', 'N': 'n_sources', 'V_sys': 'v_sys'})
by_field = by_field[['field', 'n_sources', 'ra', 'dec', 'v_sys', 'integrated_channels', 'num_integrated_channels']]
by_field[['ra', 'dec']] = by_field[['ra', 'dec']].round(5)
by_field[['n_sources']] = by_field[['n_sources']].round(0).astype(int)
by_field[['v_sys']] = by_field[['v_sys']].round(1)
by_field

,field,n_sources,ra,dec,v_sys,integrated_channels,num_integrated_channels
0,HH270VLA1,2,87.89411,2.94611,9.0,"94-100, 104-109",13
1,HOPS-12,3,83.78727,-5.93195,8.5,"92-97, 103-107",11
2,HOPS-168,2,84.07892,-6.75655,9.0,"91-98, 104-115",20
3,HOPS-173,2,84.10850,-6.41808,7.5,102,1
4,HOPS-182,3,84.07831,-6.36950,7.5,"88-93, 104-107",10
5,HOPS-193,2,84.12617,-6.02142,8.5,"97-99, 104",4
6,HOPS-203,3,84.09527,-6.76851,10.0,96-100,5
7,HOPS-213,2,85.70037,-8.66902,3.0,"92-93, 97-98",4
8,HOPS-281,2,85.10270,-7.71893,4.5,"91-94, 99-105",11
9,HOPS-282,2,85.10871,-7.62558,5.0,"93-95, 99-101",6


In [296]:
t = Table.from_pandas(by_field)
buf = StringIO()
t.write(buf, format='ascii.latex', latexdict={'tabletype': 'deluxetable'})
print(buf.getvalue())

\begin{deluxetable}
\begin{tabular}{ccccccc}
field & n_sources & ra & dec & v_sys & integrated_channels & num_integrated_channels \\
HH270VLA1 & 2 & 87.89411 & 2.94611 & 9.0 & 94-100, 104-109 & 13 \\
HOPS-12 & 3 & 83.78727 & -5.93195 & 8.5 & 92-97, 103-107 & 11 \\
HOPS-168 & 2 & 84.07892 & -6.75655 & 9.0 & 91-98, 104-115 & 20 \\
HOPS-173 & 2 & 84.1085 & -6.41808 & 7.5 & 102 & 1 \\
HOPS-182 & 3 & 84.07831 & -6.3695 & 7.5 & 88-93, 104-107 & 10 \\
HOPS-193 & 2 & 84.12617 & -6.02142 & 8.5 & 97-99, 104 & 4 \\
HOPS-203 & 3 & 84.09527 & -6.76851 & 10.0 & 96-100 & 5 \\
HOPS-213 & 2 & 85.70037 & -8.66902 & 3.0 & 92-93, 97-98 & 4 \\
HOPS-281 & 2 & 85.1027 & -7.71893 & 4.5 & 91-94, 99-105 & 11 \\
HOPS-282 & 2 & 85.10871 & -7.62558 & 5.0 & 93-95, 99-101 & 6 \\
HOPS-288 & 3 & 84.98334 & -7.50766 & 5.0 & 85-91 & 7 \\
HOPS-290 & 2 & 84.98907 & -7.49253 & 5.0 & 91-94, 99-102 & 8 \\
HOPS-304 & 2 & 85.44148 & -1.94072 & 10.0 & 94-98, 107-109 & 8 \\
HOPS-312 & 2 & 85.77371 & -1.26516 & 3.0 & 88-91, 97-98